# **MÓDULO 13**
# Projeto - Fundamentos da Descoberta de Dados

Nesse projeto trabalharemos com a base de dados de produtos de um supermercado do Chile.
A ideia é que vocês apliquem os conceitos estatísticos vistos no último módulo, mais os conceitos de visualizações de dados através de gráficos e finalizem publicando no seu github!

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

Faça a leitura dos dados do arquivo CSV:


Altere o código abaixo de acordo com seu diretório.

In [ ]:
# o arquivo está separado por vírgula (o padrão do pandas), então não precisa do delimiter=';'
df = pd.read_csv("MODULO7_PROJETOFINAL_BASE_SUPERMERCADO.csv")

df.head(10)

Os campos do nosso dataframe são:

**Title:** Nome do produto.


**Marca:** A marca do produto.


**Preco_Normal:** O preço em que o produto costuma ser vendido quando não há desconto.


**Preco_Desconto:** O preço vendido após o desconto ser aplicado.


**Preco_Anterior:** Preço em que era comercializado o produto antes do desconto aplicado.


**Desconto:** Total de desconto aplicado.






As colunas que aparecem com valores 0 são para os produtos onde não tivemos descontos aplicados.


As categorias estão em espanhol!

# 1 - Traga a média e a mediana dos preços - coluna Preco_Normal - por categoria de produto.
# Identifique as categorias que parecem ter um valor de média abaixo ou acima da mediana.

In [ ]:
# média do Preco_Normal por categoria
media = df.groupby('Categoria')['Preco_Normal'].mean()
media

In [ ]:
# mediana do Preco_Normal por categoria
mediana = df.groupby('Categoria')['Preco_Normal'].median()
mediana

# juntando tudo numa tabela só para comparar
comparacao = pd.DataFrame({'media': media, 'mediana': mediana})
comparacao['diferenca'] = comparacao['media'] - comparacao['mediana']
comparacao['media_vs_mediana'] = comparacao['diferenca'].apply(lambda x: 'média acima da mediana' if x > 0 else 'média abaixo da mediana')
comparacao.sort_values('diferenca', ascending=False)

**Categorias com média acima da mediana:** lacteos, congelados, frutas, instantaneos-y-sopas, belleza-y-cuidado-personal e verduras.

**Categoria com média abaixo da mediana:** comidas-preparadas (média ≈ 3.095 e mediana ≈ 3.290).

As diferenças mais fortes estão em **lacteos** (média ≈ 2.385 contra mediana ≈ 989), **instantaneos-y-sopas** (≈ 765 contra 439), **congelados** e **frutas**. Quando a média fica bem acima da mediana, significa que existem alguns produtos com preços muito altos puxando a média para cima.

# 2 - Traga o desvio padrão por categoria de produto.
# Qual o comportamento da média e mediana nas categorias com maior desvio?

In [ ]:
# desvio padrão do Preco_Normal por categoria
desvio = df.groupby('Categoria')['Preco_Normal'].std()
desvio = desvio.sort_values(ascending=False)
print(desvio)

# categoria com maior desvio padrão
categoria_maior_desvio = desvio.idxmax()
print('\nCategoria com maior desvio padrão:', categoria_maior_desvio)

# média e mediana das categorias, ordenadas pelo desvio
comparacao.loc[desvio.index, ['media', 'mediana']]

A categoria com maior desvio padrão é **lacteos** (≈ 3.926), ou seja, é onde os preços são mais espalhados.

Ela também é a categoria em que a **média (≈ 2.385) está muito acima da mediana (≈ 989)**, mais de 2 vezes. Isso mostra uma distribuição assimétrica, com a maior parte dos produtos baratos e alguns produtos muito caros (como os packs de 12 litros de leite) esticando a média e o desvio padrão.

O mesmo acontece em menor grau com belleza-y-cuidado-personal (2º maior desvio) e congelados (3º). Já em comidas-preparadas e verduras, que têm desvios menores, média e mediana são bem próximas.

# 3 - Plot um boxplot da distribuição do Preco_Normal para a categoria que você identificou que tem o maior desvio padrão. Como é a distribuição desses dados segundo o boxplot? Você identifica muitos outliers?

Dica: Para trazer apenas os dados da categoria que você deseja você pode usar o df.loc[df['Categoria'] == 'CATEGORIA ESCOLHIDA'

In [ ]:
# filtrando só os produtos da categoria com maior desvio padrão
df_lacteos = df.loc[df['Categoria'] == 'lacteos']

plt.figure(figsize=(8, 6))
plt.boxplot(df_lacteos['Preco_Normal'])
plt.title('Distribuição do Preço Normal - Lácteos')
plt.ylabel('Preço normal')
plt.xticks([1], ['lacteos'])
plt.savefig('boxplot_lacteos.png')
plt.show()

# contando os outliers pela regra do boxplot (1,5 x IQR)
q1 = df_lacteos['Preco_Normal'].quantile(0.25)
q3 = df_lacteos['Preco_Normal'].quantile(0.75)
iqr = q3 - q1
limite_superior = q3 + 1.5 * iqr
limite_inferior = q1 - 1.5 * iqr

outliers = df_lacteos[(df_lacteos['Preco_Normal'] > limite_superior) | (df_lacteos['Preco_Normal'] < limite_inferior)]

print('Q1:', q1, '| Q3:', q3, '| Limite superior:', limite_superior)
print('Quantidade de outliers:', len(outliers), 'de', len(df_lacteos), 'produtos')
outliers[['title', 'Marca', 'Preco_Normal']].sort_values('Preco_Normal', ascending=False).head(10)

**Distribuição (boxplot de lacteos):** metade dos produtos custa entre ≈ 269 (Q1) e ≈ 2.609 (Q3), e a mediana (≈ 989) fica mais perto da base da caixa. Isso mostra uma distribuição assimétrica à direita: muitos produtos baratos e uma "cauda" de produtos caros.

**Outliers:** sim, há muitos outliers. Pela regra do boxplot (acima de Q3 + 1,5 × IQR ≈ 6.119) são **43 produtos de 447** (cerca de 10%), todos no lado de cima. Os mais extremos são os packs de 12 unidades de leite, com preços entre ≈ 18.000 e 19.800.

# 4 - Plote um gráfico de barras onde temos a média de descontos por categoria.

In [ ]:
# média de desconto por categoria
media_desconto = df.groupby('Categoria')['Desconto'].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
plt.bar(media_desconto.index, media_desconto.values, color='steelblue')
plt.title('Média de desconto por categoria')
plt.xlabel('Categoria')
plt.ylabel('Média de desconto')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('barras_desconto.png')
plt.show()

**Análise do gráfico de barras:** congelados (≈ 154) e belleza-y-cuidado-personal (≈ 123) têm as maiores médias de desconto, seguidas de comidas-preparadas (≈ 43) e lacteos (≈ 17). Frutas, verduras e instantaneos-y-sopas têm média **zero**, ou seja, nenhum produto com desconto. Vale lembrar que a média inclui os produtos sem desconto: cerca de 89% da base não tem desconto nenhum.

# 5 - Plote um gráfico de mapa interativo agrupando os dados por categoria, marca e trazendo a média de desconto.

In [ ]:
# agrupando por categoria e marca: média de desconto e quantidade de produtos
agrupado = df.groupby(['Categoria', 'Marca']).agg(
    media_desconto=('Desconto', 'mean'),
    qtd_produtos=('title', 'count')
).reset_index()

# treemap: o tamanho do retângulo é a quantidade de produtos
# e a cor mostra a média de desconto
fig = px.treemap(
    agrupado,
    path=['Categoria', 'Marca'],
    values='qtd_produtos',
    color='media_desconto',
    color_continuous_scale='RdYlGn',
    title='Média de desconto por categoria e marca'
)
fig.update_traces(hovertemplate='<b>%{label}</b><br>Produtos: %{value}<br>Média de desconto: %{color:.0f}')
fig.write_html('mapa_interativo.html')  # salva uma versão interativa
fig.show()